# nanowhale — 1B MoE from Scratch on Colab

**All-in-one notebook.** Zero external scripts. Runs on H100 / A100 / T4.

- DeepSeek-V4 architecture (MLA + MoE + Hyper-Connections)
- ~1.2B total / ~400M active params (12 experts, top-2 routing)
- 256k context via curriculum training
- 7 reasoning-heavy public datasets
- **Auto-detects GPU and tunes batch size**

> **WIP — Do not use yet. Training stability still being refined.**

## 1. Setup & Imports

In [1]:
!pip install -q torch transformers datasets safetensors

import os, sys, math, random, json
from datetime import datetime
import re
from typing import Optional, Tuple, List

import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset, Dataset
from transformers import PreTrainedTokenizerFast
from transformers.modeling_outputs import BaseModelOutputWithPast, CausalLMOutputWithPast
from transformers.modeling_utils import PreTrainedModel
from transformers.generation import GenerationMixin
from transformers.configuration_utils import PretrainedConfig
from safetensors.torch import save_file, load_file

# ---- Fix Colab __file__ issue ----
if not hasattr(sys.modules["__main__"], "__file__"):
    sys.modules["__main__"].__file__ = "colab_notebook.py"
if not os.path.exists("colab_notebook.py"):
    open("colab_notebook.py", "w").close()

# ---- GPU info ----
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")
print(f"PyTorch: {torch.__version__}")

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 102.0 GB
PyTorch: 2.10.0+cu128


## 2. Model Architecture (DeepSeek-V4)

In [2]:
# ================================================================
# Configuration
# ================================================================

class DeepseekV4Config(PretrainedConfig):
    model_type = "deepseek_v4"
    keys_to_ignore_at_inference = ["past_key_values"]

    def __init__(self, vocab_size=129280, hidden_size=768, num_hidden_layers=16,
                 num_attention_heads=16, num_key_value_heads=1,
                 moe_intermediate_size=2048, n_routed_experts=12, n_shared_experts=2,
                 num_experts_per_tok=2, norm_topk_prob=True, scoring_func="sqrtsoftplus",
                 routed_scaling_factor=1.0, topk_method="noaux_tc", num_hash_layers=2,
                 swiglu_limit=10.0, q_lora_rank=384, head_dim=128, qk_rope_head_dim=32,
                 o_groups=4, o_lora_rank=192, sliding_window=128,
                 compress_ratios=None, compress_rope_theta=160000.0,
                 index_n_heads=64, index_head_dim=128, index_topk=512,
                 hc_mult=2, hc_sinkhorn_iters=3, hc_eps=1e-6,
                 num_nextn_predict_layers=1, hidden_act="silu",
                 max_position_embeddings=131072, initializer_range=0.02,  # 128k context capability
                 rms_norm_eps=1e-6, use_cache=True, pad_token_id=0, bos_token_id=0,
                 eos_token_id=1, tie_word_embeddings=False, rope_theta=500000.0,
                 rope_scaling=None, attention_bias=False, attention_dropout=0.0, **kwargs):
        self.vocab_size = vocab_size; self.hidden_size = hidden_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.num_key_value_heads = num_key_value_heads or num_attention_heads
        self.moe_intermediate_size = moe_intermediate_size
        self.n_routed_experts = n_routed_experts; self.n_shared_experts = n_shared_experts
        self.num_experts_per_tok = num_experts_per_tok
        self.norm_topk_prob = norm_topk_prob; self.scoring_func = scoring_func
        self.routed_scaling_factor = routed_scaling_factor
        self.topk_method = topk_method; self.num_hash_layers = num_hash_layers
        self.swiglu_limit = swiglu_limit; self.q_lora_rank = q_lora_rank
        self.head_dim = head_dim; self.qk_rope_head_dim = qk_rope_head_dim
        self.nope_head_dim = head_dim - qk_rope_head_dim
        self.o_groups = o_groups; self.o_lora_rank = o_lora_rank
        self.sliding_window = sliding_window
        self.compress_ratios = compress_ratios or [0] * (num_hidden_layers + 1)
        self.compress_rope_theta = compress_rope_theta
        self.index_n_heads = index_n_heads; self.index_head_dim = index_head_dim
        self.index_topk = index_topk; self.hc_mult = hc_mult
        self.hc_sinkhorn_iters = hc_sinkhorn_iters; self.hc_eps = hc_eps
        self.num_nextn_predict_layers = num_nextn_predict_layers
        self.hidden_act = hidden_act; self.max_position_embeddings = max_position_embeddings
        self.initializer_range = initializer_range
        self.rms_norm_eps = rms_norm_eps; self.use_cache = use_cache
        self.rope_theta = rope_theta; self.rope_scaling = rope_scaling
        self.attention_bias = attention_bias; self.attention_dropout = attention_dropout
        super().__init__(pad_token_id=pad_token_id, bos_token_id=bos_token_id,
                         eos_token_id=eos_token_id, tie_word_embeddings=tie_word_embeddings, **kwargs)


In [3]:
# ================================================================
# RMSNorm, RoPE, Hyper-Connection helpers
# ================================================================

class DeepseekV4RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        dtype = x.dtype; x = x.float()
        return (self.weight * (x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps))).to(dtype)

def precompute_freqs_cis(dim, seqlen, base):
    freqs = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
    t = torch.arange(seqlen, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    return torch.stack([freqs.cos(), freqs.sin()], dim=0)

def apply_rotary_emb(x, cos_sin):
    cos, sin = cos_sin[0], cos_sin[1]
    d = x.shape[-1] // 2; x1, x2 = x[..., :d], x[..., d:]
    while cos.ndim < x1.ndim: cos, sin = cos.unsqueeze(0), sin.unsqueeze(0)
    return torch.cat([x1 * cos + x2 * sin, x1 * (-sin) + x2 * cos], dim=-1).to(x.dtype)

def hc_split_sinkhorn(mixes, hc_scale, hc_base, hc_mult, sinkhorn_iters, eps):
    pre = torch.sigmoid(mixes[..., :hc_mult] * hc_scale[0] + hc_base[:hc_mult]) + eps
    post = 2 * torch.sigmoid(mixes[..., hc_mult:2*hc_mult] * hc_scale[1] + hc_base[hc_mult:2*hc_mult])
    comb = mixes[..., 2*hc_mult:].reshape(*mixes.shape[:-1], hc_mult, hc_mult) * hc_scale[2] + hc_base[2*hc_mult:].reshape(hc_mult, hc_mult)
    comb = F.softmax(comb, dim=-1) + eps
    comb = comb / (comb.sum(dim=-2, keepdim=True) + eps)
    for _ in range(sinkhorn_iters - 1):
        comb = comb / (comb.sum(dim=-1, keepdim=True) + eps)
        comb = comb / (comb.sum(dim=-2, keepdim=True) + eps)
    return pre, post, comb

In [4]:
# ================================================================
# MLA Attention
# ================================================================

class DeepseekV4Attention(nn.Module):
    def __init__(self, config, layer_idx):
        super().__init__()
        self.num_heads = config.num_attention_heads
        self.head_dim = config.head_dim
        self.qk_rope_head_dim = config.qk_rope_head_dim
        self.o_groups = config.o_groups
        self.o_lora_rank = config.o_lora_rank
        self.scaling = config.head_dim ** -0.5
        self.wq_a = nn.Linear(config.hidden_size, config.q_lora_rank, bias=False)
        self.q_norm = DeepseekV4RMSNorm(config.q_lora_rank)
        self.wq_b = nn.Linear(config.q_lora_rank, self.num_heads * self.head_dim, bias=False)
        self.wkv = nn.Linear(config.hidden_size, self.head_dim, bias=False)
        self.kv_norm = DeepseekV4RMSNorm(self.head_dim)
        ghd = self.num_heads * self.head_dim // self.o_groups
        self.wo_a = nn.Linear(ghd, self.o_groups * self.o_lora_rank, bias=False)
        self.wo_b = nn.Linear(self.o_groups * self.o_lora_rank, config.hidden_size, bias=False)

    def forward(self, hidden_states, attention_mask=None, position_ids=None,
                freqs_cis=None, past_key_value=None, use_cache=False):
        bsz, seqlen, _ = hidden_states.shape
        q = self.wq_b(self.q_norm(self.wq_a(hidden_states)))
        q = q.view(bsz, seqlen, self.num_heads, self.head_dim).transpose(1, 2)
        q = q * torch.rsqrt(q.float().pow(2).mean(-1, keepdim=True) + 1e-6)
        q = q.to(hidden_states.dtype)
        kv = self.kv_norm(self.wkv(hidden_states)).unsqueeze(1)
        if freqs_cis is not None:
            qr = apply_rotary_emb(q[..., -self.qk_rope_head_dim:], freqs_cis)
            kvr = apply_rotary_emb(kv[..., -self.qk_rope_head_dim:], freqs_cis)
            q = torch.cat([q[..., :-self.qk_rope_head_dim], qr], dim=-1)
            kv = torch.cat([kv[..., :-self.qk_rope_head_dim], kvr], dim=-1)
        attn = F.scaled_dot_product_attention(q, kv.expand(-1, self.num_heads, -1, -1),
            kv.expand(-1, self.num_heads, -1, -1), attn_mask=attention_mask,
            is_causal=(attention_mask is None), scale=self.scaling)
        attn = attn.transpose(1, 2).reshape(bsz, seqlen, self.o_groups, -1)
        wo_a_w = self.wo_a.weight.view(self.o_groups, self.o_lora_rank, -1)
        attn = torch.einsum("bsgd,grd->bsgr", attn, wo_a_w).flatten(2)
        return self.wo_b(attn), None

In [5]:
# ================================================================
# MoE (Experts + Gate + Full MoE Layer)
# ================================================================

class DeepseekV4Expert(nn.Module):
    def __init__(self, hidden_size, inter_size, limit=10.0):
        super().__init__()
        self.w1 = nn.Linear(hidden_size, inter_size, bias=False)
        self.w2 = nn.Linear(inter_size, hidden_size, bias=False)
        self.w3 = nn.Linear(hidden_size, inter_size, bias=False)
        self.limit = limit
    def forward(self, x):
        g, u = self.w1(x).float(), self.w3(x).float()
        if self.limit > 0: u, g = u.clamp(-self.limit, self.limit), g.clamp(max=self.limit)
        return self.w2((F.silu(g) * u).to(self.w2.weight.dtype))

class DeepseekV4Gate(nn.Module):
    def __init__(self, config, layer_idx):
        super().__init__()
        self.topk = config.num_experts_per_tok
        self.sfunc = config.scoring_func; self.scale = config.routed_scaling_factor
        self.hash = layer_idx < config.num_hash_layers
        self.weight = nn.Parameter(torch.empty(config.n_routed_experts, config.hidden_size))
        self.bias = None if self.hash else nn.Parameter(torch.zeros(config.n_routed_experts))
    def forward(self, x):
        s = F.linear(x.float(), self.weight.float())
        if self.sfunc == "sqrtsoftplus": s = F.softplus(s).sqrt()
        elif self.sfunc == "sigmoid": s = s.sigmoid()
        elif self.sfunc == "softmax": s = s.softmax(dim=-1)
        orig = s; s = s + self.bias if self.bias is not None else s
        idx = s.topk(self.topk, dim=-1)[1]
        w = orig.gather(1, idx)
        if self.sfunc != "softmax": w = w / (w.sum(dim=-1, keepdim=True) + 1e-20)
        return (w * self.scale).to(x.dtype), idx

class DeepseekV4MoE(nn.Module):
    def __init__(self, config, layer_idx):
        super().__init__()
        self.hidden = config.hidden_size; self.n_exp = config.n_routed_experts
        self.gate = DeepseekV4Gate(config, layer_idx)
        self.experts = nn.ModuleList([DeepseekV4Expert(config.hidden_size, config.moe_intermediate_size, config.swiglu_limit) for _ in range(self.n_exp)])
        self.shared = DeepseekV4Expert(config.hidden_size, config.moe_intermediate_size)
    def forward(self, x):
        sh = x.shape; xf = x.view(-1, self.hidden)
        w, idx = self.gate(xf)
        y = torch.zeros_like(xf, dtype=torch.float32)
        cnt = torch.bincount(idx.flatten(), minlength=self.n_exp)
        for i in range(self.n_exp):
            if cnt[i] == 0: continue
            r, c = torch.where(idx == i)
            y[r] += (w[r, c].unsqueeze(-1) * self.experts[i](xf[r]).float())
        return (y + self.shared(xf).float()).to(x.dtype).view(sh)

In [6]:
# ================================================================
# Transformer Block with Hyper-Connections
# ================================================================

class DeepseekV4Block(nn.Module):
    def __init__(self, config, layer_idx):
        super().__init__()
        self.hc_mult = config.hc_mult; self.neps = config.rms_norm_eps
        self.heps = config.hc_eps; self.hcit = config.hc_sinkhorn_iters
        self.attn = DeepseekV4Attention(config, layer_idx)
        self.ffn = DeepseekV4MoE(config, layer_idx)
        self.attn_norm = DeepseekV4RMSNorm(config.hidden_size)
        self.ffn_norm = DeepseekV4RMSNorm(config.hidden_size)
        mh = (2 + config.hc_mult) * config.hc_mult
        hd = config.hc_mult * config.hidden_size
        self.afn = nn.Parameter(torch.empty(mh, hd))
        self.ffn_fn = nn.Parameter(torch.empty(mh, hd))
        self.abase = nn.Parameter(torch.empty(mh))
        self.fbase = nn.Parameter(torch.empty(mh))
        self.ascale = nn.Parameter(torch.empty(3))
        self.fscale = nn.Parameter(torch.empty(3))

    def _hc_pre(self, x, fn, sc, bs):
        dt = x.dtype; xf = x.flatten(2).float()
        rs = torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.neps)
        pre, post, comb = hc_split_sinkhorn(F.linear(xf, fn.float()) * rs, sc, bs, self.hc_mult, self.hcit, self.heps)
        return (pre.unsqueeze(-1) * x.float()).sum(dim=2).to(dt), post, comb

    def _hc_post(self, x, res, post, comb):
        return (post.unsqueeze(-1) * x.unsqueeze(2).float() + torch.einsum("bsij,bsjd->bsid", comb.float(), res.float())).to(x.dtype)

    def forward(self, x, attention_mask=None, position_ids=None, freqs_cis=None, **kw):
        res = x
        y, po, co = self._hc_pre(x, self.afn, self.ascale, self.abase)
        y = self.attn(self.attn_norm(y), attention_mask=attention_mask, freqs_cis=freqs_cis)[0]
        x = self._hc_post(y, res, po, co)
        res = x
        y, po, co = self._hc_pre(x, self.ffn_fn, self.fscale, self.fbase)
        y = self.ffn(self.ffn_norm(y))
        return self._hc_post(y, res, po, co), None

In [7]:
# ================================================================
# Full Model + CausalLM Head
# ================================================================

class DeepseekV4PreTrainedModel(PreTrainedModel):
    config_class = DeepseekV4Config
    base_model_prefix = "model"
    supports_gradient_checkpointing = True
    _no_split_modules = ["DeepseekV4Block"]
    _skip_keys_device_placement = ["past_key_values"]

    def _init_weights(self, module):
        s = self.config.initializer_range
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(0, s)
            if module.bias is not None: module.bias.data.zero_()
        elif isinstance(module, nn.Embedding): module.weight.data.normal_(0, s)
        elif isinstance(module, DeepseekV4RMSNorm): module.weight.data.fill_(1.0)
        elif isinstance(module, DeepseekV4Gate):
            module.weight.data.normal_(0, s)
            if module.bias is not None: module.bias.data.zero_()
        elif isinstance(module, DeepseekV4Block):
            for n in ["afn", "ffn_fn"]: nn.init.normal_(getattr(module, n), std=0.01)
            for n in ["abase", "fbase"]: nn.init.zeros_(getattr(module, n))
            for n in ["ascale", "fscale"]: nn.init.ones_(getattr(module, n))

class DeepseekV4Model(DeepseekV4PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([DeepseekV4Block(config, i) for i in range(config.num_hidden_layers)])
        self.norm = DeepseekV4RMSNorm(config.hidden_size)
        hd = config.hc_mult * config.hidden_size
        self.hc_head_fn = nn.Parameter(torch.empty(config.hc_mult, hd))
        self.hc_head_base = nn.Parameter(torch.empty(config.hc_mult))
        self.hc_head_scale = nn.Parameter(torch.empty(1))
        self.register_buffer("freqs_cis",
            precompute_freqs_cis(config.qk_rope_head_dim, config.max_position_embeddings, config.rope_theta),
            persistent=False)
        self.gradient_checkpointing = False
        self.post_init()

    def _init_weights(self, module):
        super()._init_weights(module)
        if module is self:
            nn.init.normal_(self.hc_head_fn, std=0.01)
            nn.init.zeros_(self.hc_head_base)
            nn.init.ones_(self.hc_head_scale)

    def _hc_head(self, x):
        dt = x.dtype; xf = x.flatten(2).float()
        rs = torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.config.rms_norm_eps)
        pre = torch.sigmoid(F.linear(xf, self.hc_head_fn.float()) * rs * self.hc_head_scale.float() + self.hc_head_base.float()) + self.config.hc_eps
        return (pre.unsqueeze(-1) * x.float()).sum(dim=2).to(dt)

    def forward(self, input_ids=None, attention_mask=None, position_ids=None, inputs_embeds=None):
        if inputs_embeds is None:
            inputs_embeds = self.embed_tokens(input_ids)
        bsz, seqlen = inputs_embeds.shape[:2]
        if position_ids is None:
            position_ids = torch.arange(seqlen, device=inputs_embeds.device).unsqueeze(0)
        position_ids = position_ids.clamp(0, self.config.max_position_embeddings - 1)
        freqs_cis = self.freqs_cis[:, position_ids.squeeze(0)].to(inputs_embeds.device)
        if attention_mask is None:
            mask = torch.triu(torch.full((seqlen, seqlen), float("-inf"), device=inputs_embeds.device, dtype=inputs_embeds.dtype), diagonal=1).unsqueeze(0).unsqueeze(0)
        else:
            causal = torch.triu(torch.full((seqlen, seqlen), float("-inf"), device=inputs_embeds.device, dtype=inputs_embeds.dtype), diagonal=1).unsqueeze(0).unsqueeze(0)
            pmask = (1.0 - attention_mask) * float("-inf")
            mask = causal + pmask.unsqueeze(1).unsqueeze(2).to(dtype=inputs_embeds.dtype)
        hs = inputs_embeds.unsqueeze(2).expand(-1, -1, self.config.hc_mult, -1).contiguous()
        for layer in self.layers:
            if self.gradient_checkpointing and self.training:
                hs, _ = torch.utils.checkpoint.checkpoint(layer, hs, mask, position_ids, freqs_cis, use_reentrant=False)
            else:
                hs, _ = layer(hs, attention_mask=mask, position_ids=position_ids, freqs_cis=freqs_cis)
        return BaseModelOutputWithPast(last_hidden_state=self.norm(self._hc_head(hs)), past_key_values=None)

class DeepseekV4ForCausalLM(DeepseekV4PreTrainedModel, GenerationMixin):
    _tied_weights_keys = {"lm_head.weight": "model.embed_tokens.weight"}
    def __init__(self, config):
        super().__init__(config)
        self.model = DeepseekV4Model(config)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.post_init()
    def forward(self, input_ids=None, attention_mask=None, labels=None, **kw):
        out = self.model(input_ids=input_ids, attention_mask=attention_mask)
        logits = self.lm_head(out.last_hidden_state)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits[..., :-1, :].contiguous().view(-1, self.config.vocab_size),
                                   labels[..., 1:].contiguous().view(-1), ignore_index=-100)
        return CausalLMOutputWithPast(loss=loss, logits=logits, past_key_values=None)
    def prepare_inputs_for_generation(self, input_ids, **kw):
        return {"input_ids": input_ids[:, -1:]} if kw.get("past_key_values") else {"input_ids": input_ids}

print("Model code loaded.")

Model code loaded.


## 3. Download Tokenizer

In [8]:
!mkdir -p tokenizer
!wget -q -O tokenizer/tokenizer.json https://huggingface.co/cmpatino/nanowhale-100m/resolve/main/tokenizer.json
!wget -q -O tokenizer/tokenizer_config.json https://huggingface.co/cmpatino/nanowhale-100m/resolve/main/tokenizer_config.json

tokenizer = PreTrainedTokenizerFast.from_pretrained("tokenizer")
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer loaded. Vocab: {tokenizer.vocab_size}")

Tokenizer loaded. Vocab: 128000


## 4. Prepare Reasoning Dataset

7 public datasets: general English + code + math + long-context

In [9]:
DATASETS = [
    ("fineweb2", "HuggingFaceFW/fineweb-2",               300, "general"),
    ("c4",       "allenai/c4",                           350, "clean-web"),
    ("fffweb",   "m-a-p/FineFineWeb",                    400, "curated"),
    ("code",     "deepmind/code_contests",               128, "coding"),
    ("mbpp",     "google-research-datasets/mbpp",        64,  "python"),
    ("gsm8k",    "openai/gsm8k",                         64,  "math"),
    ("long",     "HuggingFaceFW/fineweb-2",              500, "long-ctx"),
]

MAX_PER = 10000
PACK_LONG = 49152
all_examples = []

for key, name, min_chars, purpose in DATASETS:
    print(f"[{key}] {purpose}...")
    try:
        ds = load_dataset(name, split="train", streaming=True)
    except:
        try:
            ds = load_dataset(name, "en", split="train", streaming=True)
        except:
            try:
                ds = load_dataset(name, "main", split="train", streaming=False)
            except:
                try:
                    ds = load_dataset(name, split="train", streaming=False)
                except Exception as e:
                    print(f"  SKIP: {e}")
                    continue

    samples = []
    for s in ds:
        text = s.get("text") or s.get("content") or ""
        if "question" in s:
            a = s.get("answer") or s.get("solution") or ""
            text = f"Q: {s['question']}\nA: {a}" if a else s["question"]
        if "problem" in s:
            sol = s.get("solution") or s.get("solutions") or ""
            text = f"Problem: {s['problem']}\nSolution: {sol}" if sol else s["problem"]
        if not text or len(text) < min_chars: continue
        # Quality filtering for better training efficiency
        # Skip low-quality or problematic samples
        if len(text) < 200:  # Increased minimum length
            continue
        # Basic quality checks
        if text.count('\n') < 2:  # At least some structure
            continue
        # Skip if too many special characters
        if text.count('<') > len(text) * 0.1:  # HTML tags
            continue
        if text.count('=') > len(text) * 0.05:  # Math formulas ok, but not too many
            continue
        # Quality filtering for better training efficiency
        # Skip low-quality or problematic samples
        if len(text) < 200:  # Increased minimum length
            continue
        # Basic quality checks
        if text.count('\n') < 2:  # At least some structure
            continue
        # Skip if too many special characters
        if text.count('<') > len(text) * 0.1:  # HTML tags
            continue
        if text.count('=') > len(text) * 0.05:  # Math formulas ok, but not too many
            continue
        samples.append({"text": text})
        if len(samples) >= MAX_PER: break

    if key == "long":
        packed, buf = [], ""
        for s in samples:
            buf += "\n\n" + s["text"]
            if len(buf) > PACK_LONG: packed.append({"text": buf.strip()}); buf = ""
        if buf: packed.append({"text": buf.strip()})
        print(f"  {len(packed)} packed long-context examples")
        samples = packed
    else:
        print(f"  {len(samples)} examples")
    all_examples.extend(samples)

random.shuffle(all_examples)
print(f"\nTotal: {len(all_examples):,} examples")

[fineweb2] general...
  SKIP: BuilderConfig 'default' not found. Available: ['aai_Latn', 'aak_Latn', 'aau_Latn', 'aaz_Latn', 'aba_Latn', 'abi_Latn', 'abk_Cyrl', 'abn_Latn', 'abq_Cyrl', 'abs_Latn', 'abt_Latn', 'abx_Latn', 'aby_Latn', 'abz_Latn', 'aca_Latn', 'acd_Latn', 'ace_Latn', 'acf_Latn', 'ach_Latn', 'acm_Arab', 'acn_Latn', 'acr_Latn', 'acu_Latn', 'ada_Latn', 'ade_Latn', 'adh_Latn', 'adi_Latn', 'adj_Latn', 'adl_Latn', 'ady_Cyrl', 'adz_Latn', 'aeb_Arab', 'aer_Latn', 'aeu_Latn', 'aey_Latn', 'afr_Latn', 'agd_Latn', 'agg_Latn', 'agm_Latn', 'agn_Latn', 'agr_Latn', 'agt_Latn', 'agu_Latn', 'agw_Latn', 'agx_Cyrl', 'aha_Latn', 'ahk_Latn', 'aia_Latn', 'aii_Syrc', 'aim_Latn', 'ain_Latn', 'ajg_Latn', 'aji_Latn', 'ajz_Latn', 'akb_Latn', 'ake_Latn', 'akh_Latn', 'akp_Latn', 'alj_Latn', 'aln_Latn', 'alp_Latn', 'alq_Latn', 'als_Latn', 'alt_Cyrl', 'aly_Latn', 'alz_Latn', 'ame_Latn', 'amf_Latn', 'amh_Ethi', 'ami_Latn', 'amk_Latn', 'amm_Latn', 'amn_Latn', 'amp_Latn', 'amr_Latn', 'amu_Latn', 'amx_Latn',

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

  10000 examples
[fffweb] curated...


Resolving data files:   0%|          | 0/66107 [00:00<?, ?it/s]

  10000 examples
[code] coding...


Resolving data files:   0%|          | 0/39 [00:00<?, ?it/s]

  0 examples
[mbpp] python...
  0 examples
[gsm8k] math...
  7440 examples
[long] long-ctx...
  SKIP: BuilderConfig 'default' not found. Available: ['aai_Latn', 'aak_Latn', 'aau_Latn', 'aaz_Latn', 'aba_Latn', 'abi_Latn', 'abk_Cyrl', 'abn_Latn', 'abq_Cyrl', 'abs_Latn', 'abt_Latn', 'abx_Latn', 'aby_Latn', 'abz_Latn', 'aca_Latn', 'acd_Latn', 'ace_Latn', 'acf_Latn', 'ach_Latn', 'acm_Arab', 'acn_Latn', 'acr_Latn', 'acu_Latn', 'ada_Latn', 'ade_Latn', 'adh_Latn', 'adi_Latn', 'adj_Latn', 'adl_Latn', 'ady_Cyrl', 'adz_Latn', 'aeb_Arab', 'aer_Latn', 'aeu_Latn', 'aey_Latn', 'afr_Latn', 'agd_Latn', 'agg_Latn', 'agm_Latn', 'agn_Latn', 'agr_Latn', 'agt_Latn', 'agu_Latn', 'agw_Latn', 'agx_Cyrl', 'aha_Latn', 'ahk_Latn', 'aia_Latn', 'aii_Syrc', 'aim_Latn', 'ain_Latn', 'ajg_Latn', 'aji_Latn', 'ajz_Latn', 'akb_Latn', 'ake_Latn', 'akh_Latn', 'akp_Latn', 'alj_Latn', 'aln_Latn', 'alp_Latn', 'alq_Latn', 'als_Latn', 'alt_Cyrl', 'aly_Latn', 'alz_Latn', 'ame_Latn', 'amf_Latn', 'amh_Ethi', 'ami_Latn', 'amk_Latn',

## 5. Build Model (~1.2B params)

Auto-tunes batch size based on GPU VRAM

In [10]:
# ---- Build model ----
config = DeepseekV4Config(
    vocab_size=129280, hidden_size=768, num_hidden_layers=16,
    num_attention_heads=16, num_key_value_heads=1, head_dim=128,
    qk_rope_head_dim=32, q_lora_rank=384, o_groups=4, o_lora_rank=192,
    moe_intermediate_size=2048, n_routed_experts=12, n_shared_experts=2,
    num_experts_per_tok=2, hc_mult=2, hc_sinkhorn_iters=3,
    max_position_embeddings=262144, rope_theta=500000.0,
)
model = DeepseekV4ForCausalLM(config).cuda()
total = sum(p.numel() for p in model.parameters())
print(f"Model: {total:,} params ({total/1e9:.2f}B)")

# ---- Auto-tune batch size based on VRAM ----
if vram_gb >= 70:
    BATCH_SIZE = 8; GRAD_ACCUM = 8      # H100/A100 80GB: eff batch 64
elif vram_gb >= 35:
    BATCH_SIZE = 4; GRAD_ACCUM = 8      # A100 40GB / L40S: eff batch 32
elif vram_gb >= 20:
    BATCH_SIZE = 2; GRAD_ACCUM = 12     # T4 16GB-ish / A10: eff batch 24
else:
    BATCH_SIZE = 1; GRAD_ACCUM = 16     # T4 free: eff batch 16

print(f"Batch size: {BATCH_SIZE} x {GRAD_ACCUM} grad accum = effective {BATCH_SIZE * GRAD_ACCUM}")
print("Training: 5000 steps with 8k context (Flash Attention + gradient checkpointing)")

# Batch size ramping schedule (Seesaw-inspired)
def get_batch_size(step, base_batch, max_batch=8):
    """Double batch size at 25%, 50%, 75% of training."""
    progress = step / STEPS
    if progress < 0.25:
        return base_batch
    elif progress < 0.5:
        return min(base_batch * 2, max_batch)
    elif progress < 0.75:
        return min(base_batch * 4, max_batch)
    else:
        return min(base_batch * 8, max_batch)


# ---- torch.compile ----
try:
    # Removed mode="reduce-overhead" to avoid CUDA Graphs issues with dynamic shapes/grad accum
    model = torch.compile(model)
    print("torch.compile: ON")
except Exception as e:
    print(f"torch.compile: SKIP ({e})")

# ---- Gradient checkpointing for memory ----
model.model.gradient_checkpointing = True
print("Gradient checkpointing: ON")

# ---- Optimizer ----
# Layer-wise learning rates (lower layers get smaller LR)
def apply_layer_wise_lr(model, base_lr, decay=0.95):
    """Apply decreasing LR to earlier layers."""
    param_groups = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        # Calculate layer depth (deeper = higher LR)
        layer_match = re.search(r'layers\.(\d+)', name)
        if layer_match:
            layer_idx = int(layer_match.group(1))
            total_layers = model.config.num_hidden_layers
            # Normalize layer index (0 = bottom, total-1 = top)
            normalized_depth = layer_idx / total_layers
            # LR multiplier: deeper layers get higher LR
            lr_mult = decay ** (total_layers - layer_idx - 1)
        else:
            lr_mult = 1.0  # Embedding/head layers
        param_groups.append({'params': param, 'lr': base_lr * lr_mult})
    return param_groups

# Create optimizer with layer-wise LR
param_groups = apply_layer_wise_lr(model, base_lr=3e-4)
optimizer = torch.optim.AdamW(param_groups, weight_decay=0.1)
scaler = torch.amp.GradScaler("cuda")


Model: 1,215,222,027 params (1.22B)
Batch size: 8 x 8 grad accum = effective 64
Training: 5000 steps with 8k context (Flash Attention + gradient checkpointing)
torch.compile: ON
Gradient checkpointing: ON


## 6. Training Loop

- Multi-sequence batching (maximizes GPU utilization)
- Curriculum context: 4k -> 8k -> 32k -> 64k -> 128k -> 256k
- Dynamic padding per batch (pad to longest, not fixed)
- Checkpoints every 5k steps

In [11]:
# === Training settings ===
STEPS = 5000          # Full run (set to 300 for smoke test)
LR = 3e-4
LOG_EVERY = 20
SAVE_EVERY = 5000
OUTPUT_DIR = "/content/checkpoints/nanowhale_1b"

# Curriculum: (step, seq_len). Override short if smoke test.
CURRICULUM = [(0, 4096), (1000, 8192), (3000, 32768), (8000, 65536), (15000, 131072), (25000, 262144)]
if STEPS <= 500:
    CURRICULUM = [(0, 2048)]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"{STEPS} steps | batch {BATCH_SIZE}x{GRAD_ACCUM} | LR {LR} | seq curriculum: {[l for _,l in CURRICULUM]}")
print("Estimated training time: ~2-3 hours on H100/A100, ~4-5 hours on RTX 6000")

5000 steps | batch 8x8 | LR 0.0003 | seq curriculum: [4096, 8192, 32768, 65536, 131072, 262144]
Estimated training time: ~2-3 hours on H100/A100, ~4-5 hours on RTX 6000


In [12]:
def get_cur_len(step):
    cur = CURRICULUM[0][1]
    for s, l in CURRICULUM:
        if step >= s: cur = l
    return cur

def tokenize_all(examples, max_len):
    """Tokenize all examples, truncating to max_len."""
    out = []
    for ex in examples:
        ids = tokenizer.encode(ex["text"], truncation=True, max_length=max_len)
        if len(ids) >= 8:
            out.append(ids)
    return out

def build_batch(token_ids_list, idx, batch_size):
    """Build a proper padded batch of `batch_size` sequences.
    Pads to the longest sequence in the batch (dynamic padding)."""
    seqs = []
    for i in range(batch_size):
        tid = token_ids_list[(idx + i) % len(token_ids_list)]
        seqs.append(tid[:len(tid) - 1])       # input = all but last
    # Pad to max length in batch
    max_l = max(len(s) for s in seqs)
    input_ids = torch.zeros(batch_size, max_l, dtype=torch.long)
    labels = torch.full((batch_size, max_l), -100, dtype=torch.long)
    attn_mask = torch.zeros(batch_size, max_l)
    for i, s in enumerate(seqs):
        L = len(s)
        input_ids[i, :L] = torch.tensor(s, dtype=torch.long)
        labels[i, :L] = torch.tensor(token_ids_list[(idx + i) % len(token_ids_list)][1:L+1], dtype=torch.long)
        attn_mask[i, :L] = 1.0
    return input_ids.cuda(), labels.cuda(), attn_mask.cuda()

In [ ]:
import math

def update_lr(step):
    # 10% warmup, followed by cosine decay
    warmup_steps = max(1, int(STEPS * 0.1))
    if step < warmup_steps:
        lr = LR * (step + 1) / warmup_steps
    else:
        progress = (step - warmup_steps) / max(1, (STEPS - warmup_steps))
        lr = LR * 0.5 * (1.0 + math.cos(math.pi * progress))

    for param_group in optimizer.param_groups:
        if "initial_lr" not in param_group:
            param_group["initial_lr"] = param_group["lr"]
        # Maintain the layer-wise relative learning rates
        param_group["lr"] = param_group["initial_lr"] * (lr / LR)
    return lr

model.train()
losses = []
cur_len = get_cur_len(0)
acc_loss = 0.0
micro = 0
tokenized_ids = tokenize_all(all_examples, cur_len)
batch_idx = 0

for step in range(STEPS):
    # Update learning rate
    current_lr = update_lr(step)
    # Curriculum length update
    new_len = get_cur_len(step)
    if new_len != cur_len:
        cur_len = new_len
        tokenized_ids = tokenize_all(all_examples, cur_len)
        print(f"  [Curriculum] Step {step}: seq_len -> {cur_len}")

    # Build real multi-sequence batch (not single-example grad accum!)
    input_ids, labels, attn_mask = build_batch(tokenized_ids, batch_idx, BATCH_SIZE)
    batch_idx += BATCH_SIZE

    # Mark the beginning of a step for CUDA Graphs (used by torch.compile reduce-overhead)
    if hasattr(torch.compiler, "cudagraph_mark_step_begin"):
        torch.compiler.cudagraph_mark_step_begin()

    # Use float32 as explicitly requested
    with torch.amp.autocast("cuda", dtype=torch.float32):
        out = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
        loss = out.loss / GRAD_ACCUM

    scaler.scale(loss).backward()
    acc_loss += loss.item()
    micro += 1

    if micro % GRAD_ACCUM == 0:
        scaler.unscale_(optimizer)
        # Adaptive gradient clipping
        # Compute gradient statistics
        grad_norms = []
        for p in model.parameters():
            if p.grad is not None:
                grad_norms.append(p.grad.norm().item())
        if grad_norms:
            grad_std = torch.std(torch.tensor(grad_norms)).item()
            clip_norm = max(0.5, min(2.0, grad_std * 2))
        else:
            clip_norm = 1.0
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
        scaler.step(optimizer)
        # Weight averaging for better convergence (anytime training)
        # Simple moving average of weights
        if step > STEPS * 0.5:  # Start averaging in second half
            with torch.no_grad():
                for param in model.parameters():
                    if hasattr(param, "wa"):
                        param.wa = 0.99 * param.wa + 0.01 * param.data
                    else:
                        param.wa = param.data.clone()
        # Apply averaged weights for logging/saving
        if step % LOG_EVERY == 0 and hasattr(list(model.parameters())[0], "wa"):
            with torch.no_grad():
                for param in model.parameters():
                    if hasattr(param, "wa"):
                        param.data = param.wa
        scaler.update()
        optimizer.zero_grad()
        losses.append(acc_loss * GRAD_ACCUM)
        acc_loss = 0.0

    if (step + 1) % LOG_EVERY == 0 and losses:
        avg = sum(losses[-min(len(losses), LOG_EVERY):]) / min(len(losses), LOG_EVERY)
        print(f"  Step {step+1:5d} | Loss: {avg:.4f} | Seq: {cur_len} | Batch: {BATCH_SIZE}")

    if (step + 1) % SAVE_EVERY == 0:
        ckpt = os.path.join(OUTPUT_DIR, f"step_{step+1}")
        os.makedirs(ckpt, exist_ok=True)
        save_file(model.state_dict(), os.path.join(ckpt, "model.safetensors"))
        torch.save({"optimizer": optimizer.state_dict(), "step": step+1},
                   os.path.join(ckpt, "optimizer.pt"))
        tokenizer.save_pretrained(ckpt)
        print(f"  [Saved] {ckpt}")

# Final save
final = os.path.join(OUTPUT_DIR, "final")
os.makedirs(final, exist_ok=True)
save_file(model.state_dict(), os.path.join(final, "model.safetensors"))
tokenizer.save_pretrained(final)
print("="*60)
print(f"DONE. Model -> {final}")
if losses: print(f"Final loss: {sum(losses[-100:])/len(losses[-100:]):.4f}")
print("="*60)


/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:321: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


  Step    20 | Loss: nan | Seq: 4096 | Batch: 8
  Step    40 | Loss: nan | Seq: 4096 | Batch: 8
  Step    60 | Loss: nan | Seq: 4096 | Batch: 8
  Step    80 | Loss: nan | Seq: 4096 | Batch: 8


## 7. Save to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil
src = os.path.join(OUTPUT_DIR, "final")
dst_base = "/content/drive/MyDrive/colab/nanowhale"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
dst = f"{dst_base}/checkpoint_{timestamp}"
if os.path.exists(src):
    os.makedirs(dst, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"Saved to Drive: {dst}")
else:
    print(f"No checkpoint at {src}")

## 7.1 Upload to HuggingFace Hub

Automatically creates a new repo and uploads the trained model.

In [ ]:
# Upload to HuggingFace Hub
try:
    from huggingface_hub import HfApi, login
    from pathlib import Path

    # Login to HuggingFace
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("✅ Logged into HuggingFace Hub")

        # Create API instance
        api = HfApi()

        # Get username for repo name
        try:
            user_info = api.whoami(token=hf_token)
            username = user_info["name"]
        except:
            username = "user"

        # Create repo name with timestamp
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        repo_name = f"nanowhale-1b-{timestamp}"
        repo_id = f"{username}/{repo_name}"

        # Create repository
        print(f"Creating HuggingFace repo: {repo_id}")
        api.create_repo(repo_id=repo_id, exist_ok=True, private=False)

        # Upload model files
        print("Uploading model to HuggingFace...")
        api.upload_folder(
            folder_path=src,
            repo_id=repo_id,
            commit_message=f"Upload nanowhale-1b model - Step {step+1}"
        )
        print(f"✅ Model uploaded to https://huggingface.co/{repo_id}")
    else:
        print("⚠️  HF_TOKEN not found in Colab secrets. Skipping HuggingFace upload.")
        print("To enable: Click the key icon in Colab, add a secret named 'HF_TOKEN' with your HuggingFace API token.")
except Exception as e:
    print(f"⚠️  HuggingFace upload failed: {e}")

## 8. Quick Inference Test

In [ ]:
ckpt = os.path.join(OUTPUT_DIR, "final", "model.safetensors")
if os.path.exists(ckpt):
    model.load_state_dict(load_file(ckpt), strict=False)
    model.eval()
    prompt = "def solve_quadratic(a, b, c):"
    inp = tokenizer.encode(prompt, return_tensors="pt").cuda()
    with torch.no_grad():
        out = model.generate(inp, max_new_tokens=150, temperature=0.7, do_sample=True)
    print(tokenizer.decode(out[0]))
else:
    print("No checkpoint yet — run training first")